# 第6课：面向对象编程 — 用"类"构建世界

> **学习目标**：理解 OOP 的核心思想，掌握类与对象、继承、多态

---

## 为什么需要面向对象？

假设你被要求写一个管理 1000 只狗的程序。每只狗有名字、年龄、品种，要能叫、能跑、能吃。

**过程式编程**的做法：
- 维护几十个列表：`names = ["旺财", "小白"...]`、`ages = [3, 1]...`
- 写一堆函数：`bark(name)`, `run(name)`, `eat(name, food)`
- 每次调用都小心翼翼地把数据传给函数

问题是：数据类型越来越多（狗、猫、鸟），函数越来越多，谁跟谁是一伙的？代码像一团乱麻。

**OOP 的核心思想**只有一句话：**把数据和对这些数据的操作打包在一起**。一只狗的名字和年龄是它的**属性**，叫和跑是它的**方法**。它们本来就属于同一个整体——一个"对象"。

### 餐厅类比：过程式 vs 面向对象

- **过程式** = 一个主厨从头跟到尾。他从冰箱拿食材（数据），按食谱（代码）一步一步做。客人一多，厨房就乱套了。
- **面向对象** = 厨房分成多个工作站。沙拉站有自己的食材和工具，热菜站有另一些。每个站独立运作，加新菜品只需要加新站，不用重排整个厨房。

OOP 能让大型代码库保持**有序、可扩展、易维护**。

---

## Python 对象模型：一切皆对象，类也是对象

在深入写类之前，先理解 Python 的对象模型——这对后续所有概念都至关重要。

**Python 里"一切皆对象"**：整数是对象，字符串是对象，函数是对象，甚至连类本身也是对象。

### 每个对象都有三个核心要素

1. **身份（identity）** — `id(obj)` 返回对象在内存中的唯一地址，好比身份证号
2. **类型（type）** — `type(obj)` 告诉你这是哪个类，决定了你能对这个对象做什么
3. **值（value）** — 对象存储的实际数据

```python
# 连 int 本身也是对象
x = 42
print(type(x))       # <class 'int'>
print(type(int))     # <class 'type'> —— int 本身是 type 类的实例！
print(type(type))    # <class 'type'> —— type 是自己的类，一切类型的源头
```

### `__dict__`：对象的"属性背包"

每个 Python 对象内部维护一个字典 `__dict__`，用来存储它的所有实例属性：

```python
class Dog:
    def __init__(self, name):
        self.name = name
        self.age = 1

d = Dog("旺财")
print(d.__dict__)   # {'name': '旺财', 'age': 1} —— 这是 d 的"属性背包"
```

`d.name` 的本质是 `d.__dict__['name']`。这就是为什么属性访问本质上就是字典查询——只是 Python 帮你加了几个额外的查找步骤（后面详解）。

### 属性查找链：实例 → 类 → 父类

当你写 `obj.attr` 时，Python 按以下顺序查找：

1. 从 `obj.__dict__` 找（实例属性）
2. 没找到 → 去 `type(obj).__dict__` 找（类属性、方法）
3. 还没找到 → 去父类的 `__dict__` 找（继承链）
4. 仍然没找到 → 触发 `AttributeError`

这条查找链解释了为什么实例能"看到"类属性，也解释了为什么我们一般不在实例上定义方法——方法存在类的 `__dict__` 里，所有实例共享一份，不占实例空间。

---

## self 初探：为什么 Python 要显式写 self？

你也许在其他语言（Java、C++）里写过 `this.name = name`，但那些语言不要求你写 `this` 在参数列表里。Python 不一样——**它要求你把 self 写在方法的第一个参数位置**。

这不是语法刁难，而是 Python 设计哲学的一部分：**显式优于隐式**。

```python
class Dog:
    def __init__(self, name):  # self 必须写在这里
        self.name = name

    def bark(self):            # self 必须写在这里
        return f"{self.name}: 汪汪！"

d = Dog("旺财")
# 这行代码背后，Python 做了两件事：
# 1. d.bark  → 从 d 的类（Dog）中找到 bark 函数
# 2. 把 d 自动作为第一个参数传给 bark → 等价于 Dog.bark(d)
print(d.bark())   # Python 自动把 d 传给 self
```

写 `self` 的好处：
- **函数调用透明**：`Dog.bark(d)` 和 `d.bark()` 本质相同，你随时可以用类名直接调用
- **没有魔法**：self 只是一个普通参数名（虽然约定都用 self），Python 没有为它加特殊语法糖
- **命名空间清晰**：`self.name` 明确告诉读者"这是实例属性"，而不是局部变量

> 关于 self 的深层原理（方法绑定、描述符协议），我们将在下一节详细展开。这里先记住核心：**self 就是"调用方法的那个对象"**。

---

## Python 的类型系统：鸭子类型哲学

Python 是**动态类型**语言，这一点和 Java、C++ 等静态类型语言有根本区别。

- **静态类型**（Java）：你声明变量时必须写类型 `Dog d = ...`，编译器检查类型是否匹配
- **动态类型**（Python）：变量没有类型约束，只有对象有类型。`d = Dog("旺财")` 之后 `d` 只是指向一个 Dog 对象的标签

### 鸭子类型：不关心"你是什么"，只关心"你会什么"

Python 社区有个著名的谚语：

> **"如果它走路像鸭子、叫起来像鸭子，那它就是鸭子。"**

翻译成编程术语：**不检查对象的类型，只检查对象有没有你需要的方法**。

```python
class Duck:
    def quack(self):
        return "嘎嘎嘎！"

class Person:
    def quack(self):
        return "我学鸭子叫！"

# 这两个类没有继承关系，但只要都有 quack() 方法
def make_it_quack(thing):
    return thing.quack()

print(make_it_quack(Duck()))    # 嘎嘎嘎！
print(make_it_quack(Person()))  # 我学鸭子叫！
```

这**不是偷懒**，而是一种有意为之的设计选择：
- **灵活性极高**：不需要提前设计复杂的继承体系，够用就行
- **松耦合**：调用者和被调用者之间没有继承绑定，改一个不影响另一个
- **协议化思维**：你不需要"是一个鸭子"（is-a），只需要"表现得像鸭子"（behaves-like）

### 什么时候鸭子类型会出问题？

鸭子类型的反面是：**如果对象没有你需要的方法，运行时才会报错**。

```python
def make_it_quack(thing):
    return thing.quack()

make_it_quack(42)   # 运行时才报 AttributeError: 'int' object has no attribute 'quack'
```

这没有绝对的好坏——静态类型在编译时抓 bug，动态类型在运行时给灵活性。Python 的哲学是**信任程序员**：我们默认你知道自己在做什么，出了错你负责修。后面的课程中你会学到抽象基类（ABC）可以折中这两种方式的优势。</cell id="baaa1cf1">

In [ ]:
# ================================================
# 1. 过程式 vs OOP — 写代码看看区别
# ================================================

# ---- 过程式：数据和操作分离 ----
def dog_bark(name):
    return f"{name}: 汪汪！"

def dog_info(name, age):
    return f"{name}，{age}岁"

dog1_name, dog1_age = "旺财", 3
dog2_name, dog2_age = "小白", 1

print("[过程式]")
print(dog_bark(dog1_name))
print(dog_info(dog1_name, dog1_age))
print("问题：100只狗要维护200个变量，容易搞混\n")


# ---- 面向对象：数据和方法打包 ----
class Dog:
    # __init__ 是构造方法（初始化方法），在对象被创建时自动调用
    # self 指代"即将被创建的那个对象实例"本身，即调用此方法的对象
    # 执行 Dog("旺财", 3) 时，Python 内部做了两件事：
    #   1. 调用 __new__ 在堆内存中分配空间，创建一个空对象
    #   2. 调用 __init__ 将参数和空对象（作为 self）传入，初始化实例属性
    def __init__(self, name, age):
        self.name = name  # self.name 是实例属性，存储在该对象的 __dict__ 中，每个实例独立
        self.age = age    # self.age 同理，不同 Dog 实例的 age 互不影响，各存各的

    # 实例方法的第一个参数必须是 self，调用时 d1.bark() 等价于 Dog.bark(d1)
    # Python 自动将调用者 d1 作为第一个参数传入 self，不需要调用者手动传
    def bark(self):
        return f"{self.name}: 汪汪！"

    # self 在这里指向调用 info() 的那个具体对象
    # 例如 d1.info() 中的 self 就是 d1，d2.info() 中的 self 就是 d2
    # 这就是 self 的本质："调用方法时的那个对象引用"
    def info(self):
        return f"{self.name}，{self.age}岁"

print("[面向对象]")
# d1 是一个独立的对象实例，在堆内存中拥有自己的 name 和 age 副本（存在 d1.__dict__ 中）
d1 = Dog("旺财", 3)
# d2 是另一个独立实例，拥有完全独立的内存空间，与 d1 互不干扰
d2 = Dog("小白", 1)
print(d1.bark())     # 这里的 self 就是 d1，等价于 Dog.bark(d1)，Python 自动把 d1 传进去
print(d1.info())     # self 永远指向调用者 d1，所以 info 读到的是 d1 的 name 和 age
print("d1.bark() 永远操作 d1 的数据，不会搞混")

## 类与对象 — 蓝图与实例

| 概念 | 类比 | 说明 |
|------|------|------|
| **类 (class)** | 建筑蓝图 | 定义有什么属性（数据）和方法（行为） |
| **对象 (object)** | 建好的房子 | 具体的实物，每个对象的数据独立 |
| **实例化** | 按图纸施工 | `Dog("旺财", 3)` — 用类创建对象 |

### __init__：构造方法

`__init__` 在对象**被创建时自动调用**，负责初始化数据：

```python
class Dog:
    def __init__(self, name, age):
        self.name = name  # 给新对象添加属性
        self.age = age
```

写 `Dog("旺财", 3)` 时 Python 做了两步：创建一个空对象 → 调用 `__init__` 把参数填进去。

这一步的实际工作是：
1. Python 调用 `object.__new__(Dog)` 在堆内存中分配空间，拿到一个"空的" Dog 实例，它的 `__dict__` 还是空的
2. Python 把这个空实例作为 self，连同 `"旺财"` 和 `3` 一起传入 `Dog.__init__(self, "旺财", 3)`
3. `self.name = "旺财"` 等价于 `self.__dict__["name"] = "旺财"`

---

## self 深入解析：为什么它必须是显式的？

Python 是唯一要求**在方法定义时显式写 self**的主流语言。这背后有历史原因，但更是一种设计选择。

### 方法绑定：描述符协议揭秘

当你写 `d.bark()` 时，Python 内部经历如下步骤：

```
1. d.bark → Python 在 d 的类（Dog）的 __dict__ 中找到 bark 函数
2. 发现 bark 是一个普通函数（function 对象）
3. 函数实现了"描述符协议"（有 __get__ 方法）
4. Python 调用 bark.__get__(d, Dog) → 返回一个"绑定方法"（bound method）
5. 这个绑定方法"记住"了 d 作为第一个参数（即 self）
6. 当你加括号调用 () 时，绑定方法把 d 自动传入 bark 的第一个参数
```

这就是为什么下面两种写法完全等价：

```python
d = Dog("旺财", 3)
d.bark()            # 优雅写法：Python 自动帮你绑定了 self
Dog.bark(d)         # 本质写法：你手动传 self，不做绑定
```

### 为什么 Python 不隐藏 self？

**Guido 的坚持**：Python 的创造者 Guido van Rossum 认为，显式写 self 是 Python 的优良特性。理由如下：

- **没有语法魔法**：self 就是普通参数。Java 的 `this` 是关键字，你无法在外部调用 `ClassName.method(this_obj)`，但 Python 随时可以。
- **支持函数式操作**：你可以把方法当普通函数用——`map(Dog.bark, dogs_list)` 遍历调用每个狗的 bark
- **一清二楚的命名空间**：看到 `self.name` 你就知道它是实例属性而不是局部变量，无需 `this->name` 之类的语法区分
- **装饰器不混淆**：`@classmethod`、`@staticmethod` 改变的是方法接收的第一个参数，因为 self/cls 是显式的，转换非常自然

### self 不是关键字，只是约定

```python
class Weird:
    def __init__(this, name):    # "this" 代替 "self"，语法完全合法
        this.name = name

w = Weird("测试")
print(w.name)   # "测试" —— 正常工作
```

但永远不要这样做。**self 是 Python 社区的硬约定**，就像 PEP 8 规定缩进用 4 个空格一样。所有 Python 程序员都这样写，所有工具都这样假设。打破约定只会让你的代码难以阅读和维护。

---

## 对象内部：属性查找链的完整过程

理解了 self 之后，我们来完整串一遍属性访问的流程。

当你写 `d.speak()` 时，Python 实际上进行了两次查找：

### 第一步：查找属性 `speak`
```
d.speak  →  d.__dict__ 有 "speak" 吗？ → 没有
         →  type(d).__dict__（即 Dog.__dict__）有 "speak" 吗？ → 有！（找到函数）
```

### 第二步：描述符协议决定返回值
```
如果找到的值是"描述符"（定义了 __get__、__set__ 或 __delete__）：
  → 调用 __get__ 方法获取结果（这是 property、classmethod、普通方法工作的原理）
如果不是描述符（普通值如整数、字符串）：
  → 直接返回
```

### 第三步：调用方法
```
找到绑定方法后 + () → 执行函数体，self 自动传入
```

### __dict__ 解剖：亲眼看看

```python
class Demo:
    class_attr = "我是类属性"
    
    def __init__(self):
        self.instance_attr = "我是实例属性"
    
    def method(self):
        pass

d = Demo()
print("实例 __dict__:", d.__dict__)    # {'instance_attr': '我是实例属性'}
print("类 __dict__:", Demo.__dict__)   # 一大坨，包含 class_attr、method 等
```

类 `__dict__` 里除了你定义的属性和方法，还包含 Python 自动添加的 `__module__`、`__dict__`、`__weakref__` 等内部属性。你不必关心这些——只需要知道**属性是按照优先级从实例查到类再查到父类的**。

这个机制是 Python OOP 的基石。理解了它，你就理解了为什么类属性能被所有实例共享、为什么方法不占实例的空间、为什么实例赋值同名属性会"遮蔽"类属性。</cell id="a965fde7">

In [ ]:
# ================================================
# 2. 类与对象 + 属性与方法类型
# ================================================

# ---------- 类属性 vs 实例属性 ----------
class Dog:
    # species 是类属性，存储在 Dog.__dict__（类对象的命名空间）中
    # 所有 Dog 实例共享同一份内存引用，不会随实例化而复制
    species = "犬科"

    def __init__(self, name, age):
        # name 和 age 是实例属性，存储在实例的 __dict__（实例命名空间）中
        # 每 new 一次就创建一份全新的拷贝，在堆内存中独占一份空间
        self.name = name
        self.age = age

    def bark(self):
        # self 指向调用者，self.name 从当前实例的 __dict__ 中读取属性值
        return f"{self.name}: 汪汪！"

d1 = Dog("旺财", 3)  # d1.__dict__ 包含 {"name": "旺财", "age": 3}
d2 = Dog("小白", 1)  # d2.__dict__ 包含 {"name": "小白", "age": 1}

# Python 属性查找顺序（实例属性访问时）：实例 __dict__ → 类 __dict__ → 父类 __dict__
# d1.species：d1.__dict__ 没有 species → Dog.__dict__ 有 → 返回"犬科"
print(f"类属性共享: {d1.species}, {d2.species}")
Dog.species = "犬科动物"         # 修改类的 __dict__，影响所有尚未遮蔽（shadow）此属性的实例
print(f"修改后: {d1.species}")

d1.species = "柯基"              # 在 d1 的 __dict__ 中新建 species 属性，从此遮蔽类属性
print(f"d1 自己的: {d1.species}")  # 优先从 d1.__dict__ 读到"柯基"
print(f"d2 还是类的: {d2.species}")  # d2.__dict__ 没有 species，去 Dog.__dict__ 读到"犬科动物"


# ---------- 三种方法类型 ----------
class Student:
    school = "第一中学"    # 类属性，存储在 Student.__dict__，所有实例共享
    _count = 0            # 类属性，_count 前导下划线是约定：表示"内部实现，外部不应直接修改"

    def __init__(self, name, age):
        self.name = name  # 实例属性，存当前对象的 __dict__
        self.age = age    # 同上
        Student._count += 1  # 通过类名直接访问并修改类属性，每创建一个实例就加 1

    # 1. 实例方法 — 第一个参数 self，Python 自动传入调用者实例
    # self 指向调用 introduce() 的那个具体实例对象（如 s1）
    def introduce(self):
        # self.school：实例 __dict__ 没有 school → Student.__dict__ 找到"第一中学"
        return f"我叫{self.name}，来自{self.school}"

    # 2. 类方法 — @classmethod 装饰器将普通方法变为类方法
    # 背后原理：@classmethod 将方法包装在 classmethod 描述符对象中
    # 当 Student.total_students() 被调用时，Python 调用该描述符的 __get__ 方法
    # 然后将"类本身（Student）"作为第一个参数绑定到 cls，而非实例
    @classmethod
    def total_students(cls):
        # cls 就是 Student 类对象本身（等价于 Student），可访问类属性但不能访问实例属性
        # 即使子类调用，cls 也指向子类（体现多态），这是硬编码 Student._count 做不到的
        return f"全校共 {cls._count} 名学生"

    # 3. 静态方法 — @staticmethod 装饰器
    # 背后原理：@staticmethod 将原始函数包装在 staticmethod 描述符对象中
    # 调用时不传递任何额外参数（既不传 self 也不传 cls）
    # 本质上就是把一个普通函数"挂"在类的命名空间下方便组织，行为与独立函数完全一致
    @staticmethod
    def is_valid_age(age):
        # 没有 self 或 cls 参数，无法访问任何类属性或实例属性
        # 适合工具函数：与类逻辑相关但不需要类数据
        return 0 < age < 150

s1 = Student("小明", 16)
s2 = Student("小红", 17)
# s1.introduce() 中 self 就是 s1，等价于 Student.introduce(s1)
print(f"\n{s1.introduce()}")
# Student.total_students() → cls 被绑定为 Student 类对象
print(Student.total_students())      # 输出：全校共 2 名学生
# Student.is_valid_age(200) → 不传 self/cls，直接像普通函数一样调用
print(f"年龄合法? {Student.is_valid_age(200)}")  # 静态方法：False

### 类属性 vs 实例属性

| 特性 | 类属性 | 实例属性 |
|------|--------|---------|
| 定义 | 直接在类里赋值 | `self.xxx =` 在 `__init__` 中 |
| 归属 | 属于类 | 属于每个实例 |
| 共享 | 所有实例共享 | 每份独立 |
| 修改影响 | 影响所有尚未覆盖的实例 | 只影响当前实例 |

### 属性查找链再梳理

有了前面的基础，我们再来细看属性访问的完整路径。当 `d1.species` 被执行时：

```
1. d1.__dict__ 中存在 "species" 吗？
   → 如果 d1 创建后执行过 d1.species = "柯基"，则 d1.__dict__["species"] = "柯基"，返回
   → 否则，没找到，继续

2. type(d1).__dict__（即 Dog.__dict__）中存在 "species" 吗？
   → 如果 Dog 类定义了 species = "犬科"，则 Dog.__dict__["species"] = "犬科"，返回
   → 否则，没找到，继续

3. 遍历 Dog 的所有父类的 __dict__
   → 父类也没找到 → 抛出 AttributeError
```

这条链**只读时**是这样走的。但**赋值**不一样：

```python
d1.species = "柯基"   # 从来不走查找链，直接写到 d1.__dict__ 中
```

这就是为什么给实例赋值同名属性会"遮蔽"类属性：赋值操作直接在实例的字典里新建一个键，后续读取时优先在实例 `__dict__` 找到它，不再去类的 `__dict__` 找了。这叫作**遮蔽（shadowing）**。

### 三种方法速记

1. **实例方法** — 第一个参数 `self`，能访问实例和类的任何东西。最常用。
2. **类方法** — 加 `@classmethod`，第一个参数 `cls`（类本身）。适合操作类属性（如统计实例数量）或提供替代构造方法。
3. **静态方法** — 加 `@staticmethod`，不需要 `self` 或 `cls`。本质就是放在类里的普通函数，和类逻辑相关但不需要访问类数据。

### 描述符协议：@classmethod 和 @staticmethod 的本质

三种方法的区别背后是**描述符协议**在起作用。描述符是 Python 中一个强大的协议——任何定义了 `__get__` 方法的对象都是描述符。

**普通函数作为描述符**：
```python
def method(self):
    pass

# 访问 obj.method 时，触发 function.__get__(obj, type(obj))
# 返回"绑定方法"——一个把 obj 记住的包装对象
```

**@classmethod 包装后的函数**：
```python
# 访问 MyClass.class_method 时
# 触发 classmethod.__get__(None, MyClass)
# 返回"绑定方法"，但绑定的是类（MyClass）而不是实例
```

**@staticmethod 包装后的函数**：
```python
# 访问 MyClass.static_method 时
# 触发 staticmethod.__get__(None, MyClass)
# 返回原始函数，不做任何绑定，也不传额外参数
```

你可以这样理解：
```
实例方法：d.bark  →  绑定到实例 d，自动传 d 作为 self
类方法：  Cls.method  →  绑定到类 Cls，自动传 Cls 作为 cls
静态方法：Cls.util  →  不绑定，直接返回原始函数
```

这就是为什么类方法通过 `self.__class__` 也能调用（虽然不常见），以及为什么实例调用类方法时自动传入的是**类**而不是实例：描述符绑定的是类对象，不是实例对象。</cell id="5f3ab3d2">

## @property — 方法伪装成属性

用 @property 装饰的方法可以**像属性一样访问**（不用加括号），但背后有逻辑：

**3 个典型用途：**
1. **getter 只读控制** — 用 `@property` 暴露内部 `_xxx`，外部只能读不能直接改
2. **setter 验证** — 用 `@xxx.setter` 在赋值时检查值是否合法（如半径不能为负）
3. **计算属性** — 不存值，每次访问实时计算（如面积 = πr²）

### 为什么用 `_radius` 而不是 `radius`？

Python 没有真正的"私有"关键字。用**单下划线开头**是约定：表示"这个属性是内部实现细节，外部不要直接访问"。你想要读写都应该通过 `@property` 提供的接口。

### @property 背后的描述符协议

很多初学者用 @property 只是为了不用加括号，但它的机制远比"省括号"深刻。

@property 本质上是一个**描述符类**（property 类）的语法糖：

```python
# 你写的：
class Circle:
    @property
    def radius(self):
        return self._radius

# 等价于：
class Circle:
    def radius(self):
        return self._radius
    radius = property(radius)  # property 是一个类，接收 getter 函数
```

`property` 是 Python 内置的描述符类，它定义了三个描述符方法：

- **`__get__`** — 当你读取 `c.radius` 时触发，执行 getter 函数返回 `self._radius`
- **`__set__`** — 当你赋值 `c.radius = 10` 时触发，执行 setter 函数（如果没定义 setter，抛 `AttributeError` 阻止赋值）
- **`__delete__`** — 当你 `del c.radius` 时触发，执行 deleter 函数（如果没定义 deleter，抛异常）

因为 property 完全符合"数据描述符"（同时定义了 `__get__` 和 `__set__`），它在属性查找链中**优先级高于实例 `__dict__`**。这意味着：

```python
# 即使实例 __dict__ 中有 "radius"，property 仍然拦截赋值和读取
# 这是普通属性遮蔽的反向——property 比实例属性"权力更大"
```

这就是为什么使用 @property 时，你必须用 `self._radius`（带下划线）来存储实际数据：**`self.radius` 这个名称已经被 property 占用了**，如果你在 `__init__` 里写 `self.radius = radius`，它会触发 setter（前提是你定义了 setter），而不是直接写 `__dict__`。

### _prefix 约定：保护内部状态

Python 不像 Java 有 `private`、`protected`、`public` 关键字。Python 程序员靠约定来区分"公开接口"和"内部实现"：

| 命名 | 含义 | 示例 |
|------|------|------|
| `name` | 公开属性 / 接口 | `c.radius` 通过 property 公开 |
| `_name` | 内部实现，外部不应直接访问 | `c._radius` 存储原始值 |
| `__name` | 名字重整（name mangling），防止子类意外覆盖 | 少见，后面会提到 |

`_radius` 前的下划线只是一个约定——技术上你仍然可以 `c._radius = -999` 绕过验证。Python 的哲学是**"我们都是成年人"（We are all consenting adults）**：它相信你不会故意绕过接口搞破坏。但如果你真的绕过了，后果自负。

如果你想要更强的"保护"，可以使用双下划线 `__radius`，Python 会对它做**名字重整（name mangling）**，在内部改成 `_ClassName__radius`。但这主要用在继承场景防止属性冲突，而不是实现访问控制——大多数时候，单下划线 + 文档说明就够了。

### 计算属性：不存值，只算值

`@property` 还可以用来定义"计算属性"——不占用存储空间，每次访问时实时从已有数据计算得出：

```python
class Circle:
    def __init__(self, radius):
        self._radius = radius
    
    @property
    def area(self):
        return math.pi * self._radius ** 2  # 每次实时算
```

`c.area` 看起来像属性，背后却在做计算。好处是：
- **没有一致性问题**：数据源变了，下次访问自动拿到新结果
- **节省空间**：不需要额外字段存储
- **接口稳定**：未来你升级为缓存计算结果，外部调用代码不用改

### 完整示例：getter + setter + deleter

```python
class Temperature:
    def __init__(self, celsius):
        self._celsius = celsius      # 实际存在 _celsius 中

    @property
    def celsius(self):
        """摄氏度 getter"""
        return self._celsius

    @celsius.setter
    def celsius(self, value):
        """摄氏度 setter：验证不低于绝对零度"""
        if value < -273.15:
            raise ValueError("温度不能低于绝对零度")
        self._celsius = value

    @property
    def fahrenheit(self):
        """华氏度 —— 计算属性，只读"""
        return self._celsius * 9/5 + 32

t = Temperature(25)
print(t.celsius)      # 25 (getter)
print(t.fahrenheit)   # 77.0 (计算属性)
t.celsius = 30        # setter 验证通过
t.celsius = -300      # ValueError!
```

在这个例子中，`celsius` 是带有验证的属性，`fahrenheit` 是只读计算属性。外部代码只需 `t.celsius` 和 `t.fahrenheit`，完全不需要知道内部 `_celsius` 的存在。

In [ ]:
# ================================================
# 3. @property — 控制属性访问
# ================================================

import math

class Circle:
    def __init__(self, radius):
        # _radius 前导下划线是 Python 命名约定，表示"内部实现细节，外部不应直接访问"
        # 这不是语言强制私有（没有真正 private 关键字），全靠约定自律
        # 外部代码应该通过 @property 提供的属性接口来读写，而非直接操作 _radius
        self._radius = radius

    @property
    def radius(self):
        """getter：读半径"""
        # @property 背后的原理：将 radius 方法包装在 property 描述符对象中
        # property 实现了 Python 描述符协议（定义了 __get__ 方法）
        # 当访问 c.radius 时，Python 调用 property.__get__() → 执行这里定义好的 getter 逻辑
        return self._radius

    @radius.setter
    def radius(self, value):
        """setter：写半径，带验证"""
        # @radius.setter 背后的原理：property 描述符还定义了 __set__ 方法
        # 当执行 c.radius = 10 时，Python 调用 property.__set__() → 执行这里的 setter 逻辑
        # setter 让赋值操作可以被"拦截"并验证，这是裸属性做不到的
        # 普通 self.radius = x 没有验证能力，setter 让封装成为可能
        if value <= 0:
            raise ValueError("半径必须大于 0")
        self._radius = value

    @property
    def area(self):
        """计算属性：面积，不存值，现算"""
        # 只有 @property 没有定义对应的 setter → 只读属性，赋值会抛出 AttributeError
        # 每次访问 c.area 都实时重新计算值，不占用额外存储空间
        # 适合"从已有数据派生的值"
        return math.pi * self._radius ** 2

    @property
    def circumference(self):
        """计算属性：周长"""
        return 2 * math.pi * self._radius

c = Circle(5)
# c.radius 触发 @property getter，返回 self._radius = 5
print(f"半径: {c.radius}")
# c.area 触发 @property getter，实时计算 π * 5²
print(f"面积: {c.area:.2f}")
print(f"周长: {c.circumference:.2f}")

# c.radius = 10 触发 @radius.setter，先验证 10 > 0（通过），再赋值 self._radius = 10
c.radius = 10
print(f"新半径: {c.radius}")
# self._radius 变了，area 自动更新为新结果（不存值所以没有一致性问题）
print(f"新面积: {c.area:.2f}")

# 验证生效
try:
    c.radius = -5                     # 触发 setter，-5 <= 0 验证失败，抛出 ValueError
except ValueError as e:
    print(f"\n错误: {e}")

## 继承 — "is-a" 关系

继承的核心价值是**代码复用**。当你说"狗**是**一种动物"、"猫**是**一种动物"时，你就在描述继承关系。

- **父类（基类）** 放通用代码 → 所有动物都有名字、都会发出声音
- **子类（派生类）** 继承父类全部能力 + 自己的特化 → 狗汪汪、猫喵喵

Python 支持多继承，这意味着一个子类可以有多个父类。这带来了强大的灵活性，也引入了一些需要理解的概念。

### super() — 调用父类方法

子类的 `__init__` 通常需要先调用父类的 `__init__` 来初始化继承来的属性，再处理自己的属性。`super()` 就是干这个的。

### 方法重写（override）

子类重新定义父类已有的方法。接口不变（方法名相同），实现变了（具体逻辑不同）。

## 多态 — 同一接口不同行为

不同类的对象对**同一个方法名**做出不同的响应：

```python
for a in [Dog("旺财"), Cat("咪咪")]:
    print(a.speak())  # 同样的 .speak()，不同的结果
```

## 鸭子类型

Python 不强制检查类型。只要一个对象有你需要的方法，你就可以调用它。

> "如果它走路像鸭子、叫起来像鸭子，那它就是鸭子。"

这意味着你不需要刻意搭建复杂的继承体系。够用就行。

---

## MRO（方法解析顺序）与 C3 线性化

当类之间存在继承关系（尤其是多继承），Python 需要决定**按什么顺序查找方法**。这个顺序叫 MRO（Method Resolution Order）。

### 单继承的 MRO

```python
class Animal: pass
class Dog(Animal): pass

# Dog → Animal → object
print(Dog.__mro__)
# (<class '__main__.Dog'>, <class '__main__.Animal'>, <class 'object'>)
```

单继承时 MRO 很简单：子类在前，父类在后，直到 object。

### 多继承与钻石问题

当两个父类都有一个同名方法，子类该用谁的？更麻烦的是**钻石继承**（菱形继承）：

```python
class A:
    def say(self):
        return "A"

class B(A):
    def say(self):
        return "B"

class C(A):
    def say(self):
        return "C"

class D(B, C):  # 同时继承 B 和 C
    pass
```

```
    A
   / \
  B   C
   \ /
    D
```

D 继承自 B 和 C，B 和 C 都继承自 A。如果调用 `D().say()`，该用 B 的还是 C 的？是 B → A → C 还是 B → C → A？

Python 用 **C3 线性化算法**来计算 MRO，它的核心原则是：
1. **子类优先于父类**
2. **父类声明顺序优先**（`class D(B, C)` 中 B 先于 C）
3. **单调性**：如果 A 在 B 之前，那么在任何子类中 A 都在 B 之前

```python
print(D.__mro__)
# D → B → C → A → object
```

所以 `D().say()` 会先找 D.__dict__（没有）→ 再找 B.__dict__（有！返回 "B"）。C 的 say() 被跳过了，因为 B 优先于 C。

### `super()` 不一定是"父类"

很多人以为 `super()` 就是"调用父类的方法"。其实不准确——`super()` 是按 **MRO 查找下一个类**。

```python
class A:
    def __init__(self):
        print("A.__init__")

class B(A):
    def __init__(self):
        print("B.__init__ 开始")
        super().__init__()     # 按 MRO 找 B 的下一个 → A
        print("B.__init__ 结束")

class C(A):
    def __init__(self):
        print("C.__init__ 开始")
        super().__init__()     # 按 MRO 找 C 的下一个 → A
        print("C.__init__ 结束")

class D(B, C):
    def __init__(self):
        print("D.__init__ 开始")
        super().__init__()     # 按 MRO 找 D 的下一个 → B
        print("D.__init__ 结束")

d = D()
# 输出顺序：
# D.__init__ 开始
# B.__init__ 开始
# C.__init__ 开始
# A.__init__
# C.__init__ 结束
# B.__init__ 结束
# D.__init__ 结束
```

**关键洞察**：当调用 `B.__init__` 中的 `super().__init__()` 时，`super()` 不是"去 B 的父类 A"，而是"去 MRO 中 B 的下一个类"——也就是 C！这就是 C3 线性化的力量：**`super()` 在整个 MRO 链上协作**，确保每个类的初始化只跑一次。

这就是为什么 Python 的 `super()` 在多重继承下能正确工作，而硬编码父类名（`A.__init__(self)`）会破坏协作顺序。

### `super()` 的底层实现

```python
super().__init__(name)
# 等价于：
super(D, self).__init__(name)   # Python 3 中可省略参数
```

`super()` 接收两个参数：当前类（`D`）和当前实例（`self`）。它从 `type(self).__mro__`（即 `D.__mro__`）中找到 `D` 的位置，然后返回**下一个类**的代理对象。

```python
# D.__mro__ = (D, B, C, A, object)
# super(D, self) → 在 MRO 中找到 D，返回它后面的 B 的代理
# super(B, self) → 在 MRO 中找到 B，返回它后面的 C 的代理
# super(C, self) → 在 MRO 中找到 C，返回它后面的 A 的代理
```

这就是 MRO 加 super() 的完整图景：**`super()` 不是"父类"，而是"MRO 中的下一个"**。

### 什么时候用多继承？

多继承虽然强大，但容易让代码难以理解。实用建议：
- **更多用组合（composition）**：把功能拆到独立的类中，通过实例变量引用，而不是通过继承
- **用 Mixin 模式**：Mixins 是专门提供单一功能的小类，通过多继承组合到主类中
- **避免深层次的多继承**：超过 3 层的多继承几乎总是可以重构的

```python
class LogMixin:
    """Mixin：为类添加日志功能"""
    def log(self, message):
        print(f"[{self.__class__.__name__}] {message}")

class SaveMixin:
    """Mixin：为类添加保存功能"""
    def save(self):
        print(f"数据已保存: {self}")

class User(LogMixin, SaveMixin):
    def __init__(self, name):
        self.name = name

    def __repr__(self):
        return f"User({self.name})"

u = User("小明")
u.log("创建用户")    # LogMixin 提供
u.save()             # SaveMixin 提供
```

Mixin 模式的好处是每个 Mixin 只做一件事，职责单一，互不干扰。

---

## Python 的类型系统回顾：为什么鸭子类型能工作？

前面我们在第一节课介绍了鸭子类型，这里我们结合继承再深入一层。

Python 的类型检查发生在**运行时**而不是编译时。这意味着：

```python
def process(animal):
    return animal.speak()

# 传入什么类型都可以——只要它有 speak()
process(Dog("旺财"))    # OK
process(Duck())         # OK（没有继承关系也可以）
process(42)             # 运行时才报错
```

### 协议（Protocol）：Python 版的"接口"

其他语言有 `interface` 关键字（Java）或抽象类（C++）。Python 没有显式的接口语法，取而代之的是"协议"（protocol）——这是一个松散的约定：**实现了特定魔术方法的类自动遵循某个协议**。

比如：
- 实现 `__len__` → 你的类遵循"长度协议"，可以被 `len()` 调用
- 实现 `__iter__` 和 `__next__` → 你的类遵循"迭代协议"，可以用在 `for` 循环中
- 实现 `__getitem__` → 你的类遵循"序列协议"，可以像列表一样索引

这些协议就是鸭子类型的高级体现：**Python 不检查你的类是否继承自某个"可迭代"基类，它只检查你有没有 `__iter__` 这个东西**。

### 什么时候鸭子类型不够用？

1. **大型项目需要类型文档**：纯鸭子类型下，调用者不知道一个参数需要哪些方法。此时类型注解（type hints）可以帮大忙
2. **需要明确的接口契约**：如果团队很大，用抽象基类（ABC）定义显式接口能让沟通成本更低
3. **性能敏感场景**：运行时类型错误难以在测试覆盖不到的地方被发现，可能导致线上事故

好在 Python 提供了 `typing` 模块和 `abc` 模块，让你在需要的时候"加一些静态类型的味道"——但这些是进阶内容，打好当前的基础再学它们会事半功倍。</cell id="862e2811">

In [ ]:
# ================================================
# 4. 继承 + 多态 + 鸭子类型
# ================================================

# ---------- 继承与 super() ----------
class Animal:
    def __init__(self, name):
        # 父类初始化自己的实例属性，供所有子类继承使用
        self.name = name

    def speak(self):
        # 父类定义接口契约：所有子类必须实现 speak()
        # 如果子类忘记实现，调用时会直接抛出 NotImplementedError
        # 这是"按约定编程"（Programming by Contract）的体现
        raise NotImplementedError("子类必须实现")

class Dog(Animal):
    # Dog 继承自 Animal → Dog 自动获得 Animal 的所有属性（name）和方法（speak 接口）
    # MRO（方法解析顺序，Method Resolution Order）：Dog → Animal → object
    # Python 用 C3 线性化算法计算 MRO，可以通过 Dog.__mro__ 查看
    def speak(self):
        # 覆盖（override）了父类的 speak 方法
        # 调用 d.speak() 时，Python 按 MRO 在 Dog 类中优先找到此方法，不会再去 Animal 查找
        return f"{self.name}: 汪汪！"

class Cat(Animal):
    # Cat 继承自 Animal，MRO：Cat → Animal → object
    def speak(self):
        # 与 Dog 不同的覆盖实现 — 相同接口不同行为
        return f"{self.name}: 喵喵~"

# 子类自动继承父类的 __init__，所以 Dog("旺财") 和 Cat("咪咪") 可以直接传 name 创建
# a 在每次循环中绑定不同的对象类型，但 .speak() 调用总是正确的
for a in [Dog("旺财"), Cat("咪咪")]:
    print(a.speak())  # 相同的 .speak() 接口得到不同的结果 — 这就是多态

# super() 调用父类
class Pet(Animal):
    # Pet 继承自 Animal，MRO：Pet → Animal → object
    def __init__(self, name, owner):
        # super() 背后的原理：返回一个 super 代理对象
        # 这个代理对象负责按 MRO 自动找到 Pet 的父类（Animal）并调用其方法
        # super().__init__(name) 等价于 Animal.__init__(self, name)
        # 作用：让父类初始化它"认识"的属性（name），避免子类重复写 self.name = name
        # 如果不调 super().__init__，Animal.__init__ 中的 self.name = name 不会执行
        super().__init__(name)   # 显式调用父类 __init__，把 self.name 交给 Animal 负责
        self.owner = owner       # 子类独有的属性，父类不知晓也不关心

    def info(self):
        # self.name 虽然是在 Animal.__init__ 中赋的，但从 Pet 实例照样能访问
        return f"{self.name} 的主人: {self.owner}"

p = Pet("小黄", "小明")
print(p.info())

# ---------- 多态 ----------
def make_sound(animal):
    # 不关心参数的实际类型，只关心对象有没有 speak() 方法
    # Python 在运行时动态查找（duck typing），没有编译时类型检查
    return animal.speak()

# 列表中的对象类型不同（Dog, Cat, Pet），但都有 speak() 方法
# 这就是"多态"的字面含义：同一函数名，不同对象形态，不同行为
for a in [Dog("大黄"), Cat("花花"), Pet("啾啾", "小红")]:
    print(make_sound(a))  # 各自决定怎么叫

# ---------- 鸭子类型 ----------
class Duck:
    # Duck 没有继承自 Animal，和 Animal、Dog 没有任何继承关系
    def speak(self):
        return "嘎嘎嘎！"

class Robot:
    # Robot 和 Duck 也毫无继承关系，但碰巧也有同名的 speak 方法
    def speak(self):
        return "哔——鸭子模式启动"

# 没有继承关系，但有同样的方法名 → 一样可以传给 make_sound()
# Python 不检查对象类型，只检查对象在运行时刻有没有所需的方法
# 谚语："如果它走路像鸭子、叫起来像鸭子，那它就是鸭子"
# 这意味着你不必为了代码复用刻意搭建复杂的继承层级，够用就好
print(make_sound(Duck()))
print(make_sound(Robot()))

## 魔术方法 — 让对象像原生类型

魔术方法（dunder methods）是那些以**双下划线开头和结尾**的方法。它们让自定义对象能用 Python 原生的运算符和函数。

### 最常见的魔术方法

| 方法 | 触发时机 | 作用 |
|------|---------|------|
| `__init__(self, ...)` | `Obj(...)` 创建时 | 初始化 |
| `__str__(self)` | `print(obj)` | 给用户看，漂亮 |
| `__repr__(self)` | 交互环境显示 | 给开发者看，应能重建对象 |
| `__add__(self, o)` | `obj + o` | 自定义加法 |
| `__sub__(self, o)` | `obj - o` | 自定义减法 |
| `__mul__(self, o)` | `obj * o` | 自定义乘法 |
| `__eq__(self, o)` | `obj == o` | 相等判断 |
| `__len__(self)` | `len(obj)` | 长度 |
| `__abs__(self)` | `abs(obj)` | 绝对值/模长 |

定义了这些后，你的类用起来就像 Python 内置类型一样自然。

### __str__ vs __repr__

- **`__str__`** → 用户友好。`print(acc)` → "小明 余额: ¥1000"
- **`__repr__`** → 开发者友好。最好能复制粘贴重建对象：`BankAccount('小明', 1000)`

---

## 魔术方法的本质：Python 的"协议系统"

魔术方法不是普通的语法糖——它们是 Python 对象模型的**基石**。每一个 Python 原生操作符和内置函数背后，都对应着一个魔术方法。

### Python 如何调度魔术方法

当你写 `obj + other` 时，Python 不是直接做加法，而是走一套完整的查找流程：

```
1. Python 查找 type(obj).__add__(obj, other)
2. 如果 type(obj) 没有定义 __add__：
   → 查找 type(other).__radd__(other, obj)（反向操作）
3. 如果两者都没有 → 抛出 TypeError
```

这不是伪装——整个 Python 语言就是用这些魔术方法构建的：

```python
# 这些操作符本质上都是函数调用的语法糖
3 + 5          →  int.__add__(3, 5)
"ab" + "cd"    →  str.__add__("ab", "cd")
[1,2] + [3,4]  →  list.__add__([1,2], [3,4])
len([1,2,3])   →  [1,2,3].__len__()
str(42)        →  (42).__str__()
```

函数调用 `obj()` 的底层也在调用魔术方法：

```python
class Callable:
    def __call__(self):
        return "我被调用了！"

c = Callable()
c()    # 等价于 c.__call__()
```

### 控制属性访问的魔术方法

| 方法 | 触发时机 |
|------|---------|
| `__getattr__(self, name)` | 属性通过正常查找链找不到时触发 |
| `__getattribute__(self, name)` | **每次**属性访问都触发（优先级最高） |
| `__setattr__(self, name, value)` | 每次属性赋值时触发 |
| `__delattr__(self, name)` | `del obj.attr` 时触发 |

```python
class TraceAttr:
    def __init__(self):
        self.value = 42

    def __getattribute__(self, name):
        print(f"正在读取属性: {name}")
        return super().__getattribute__(name)  # 必须调用父类实现

    def __setattr__(self, name, value):
        print(f"正在设置属性: {name} = {value}")
        super().__setattr__(name, value)

t = TraceAttr()
# t.value  → 打印 "正在读取属性: value"
# t.value = 99  → 打印 "正在设置属性: value = 99"
```

### 运算符重载的思维方式

很多初学者把运算符重载当作"花哨的语法糖"。但换一个角度看，**运算符重载是让你的类融入 Python 语言的关键**。

什么样的类应该定义 `__add__`？

- **数值类**：Vector、Matrix、Complex → `v1 + v2` 顺理成章
- **集合类**：自定义列表、集合 → `c1 + c2` 表示合并
- **路径类**：Path / URL → `/home` + `/user` 表示路径拼接

什么样的类**不应该**定义 `__add__`？

- **按业务逻辑不应该相加的东西**：两个 Employee 相加没有意义
- **语义模糊的操作**：如果 `user + group` 表示"加入群组"，那 `user - group` 退群？用方法名 `user.join(group)` 比运算符更清晰

**通用原则**：运算符重载应该让你的代码更**自然地可读**，而不是更花哨。当你犹豫时，使用方法（`.add()`、`.merge()`）比运算符更安全。

### 完整的魔术方法协议速查

Python 的魔术方法涵盖了方方面面，这里列出最常见的协议：

| 协议 | 关键方法 | 用途 |
|------|---------|------|
| **字符串表示** | `__str__`, `__repr__`, `__format__` | 对象如何展示为字符串 |
| **数值运算** | `__add__`, `__sub__`, `__mul__`, `__truediv__`, `__floordiv__`, `__mod__`, `__pow__` | 加减乘除等 |
| **比较运算** | `__eq__`, `__ne__`, `__lt__`, `__le__`, `__gt__`, `__ge__` | ==, !=, <, <=, >, >= |
| **类型转换** | `__int__`, `__float__`, `__bool__`, `__str__`, `__bytes__` | int(), float(), bool() 等 |
| **容器协议** | `__len__`, `__getitem__`, `__setitem__`, `__delitem__`, `__contains__`, `__iter__` | 让对象像列表/字典 |
| **属性访问** | `__getattr__`, `__setattr__`, `__delattr__`, `__getattribute__` | 属性访问钩子 |
| **上下文管理** | `__enter__`, `__exit__` | `with obj:` 语法支持 |
| **可调用对象** | `__call__` | 让实例可以像函数一样被调用 |
| **哈希** | `__hash__` | 让对象可以作为字典键或放入集合 |

你不需要一次性记住全部。关键是理解这个模式：**Python 的内置操作最终都会调用对象的魔术方法**。当你觉得"要是我的对象能支持这个操作就好了"的时候，查一下对应的魔术方法名，实现它即可。</cell id="888f8ad0">

In [ ]:
# ================================================
# 5. 魔术方法 — 让 Vector 支持 + - ==
# ================================================

class Vector:
    def __init__(self, x, y):
        # __init__ 是构造方法，当执行 Vector(3, 4) 时自动被 Python 调用
        # self 指向新创建的那个 Vector 实例
        self.x, self.y = x, y  # 实例属性，存在实例的 __dict__ 中

    def __str__(self):
        # __str__ 被 print()、str()、format() 以及 f-string 隐式调用
        # print(v1) 内部会调用 str(v1)，str(v1) 调用 v1.__str__()
        # 应该返回对"终端用户友好"的可读描述
        """print() 调用"""
        return f"({self.x}, {self.y})"

    def __repr__(self):
        # __repr__ 在交互式环境直接输入变量名回车时调用，也被 repr() 调用
        # 最佳实践：__repr__ 应返回一个字符串，让 eval(repr(obj)) 能重建对象
        # 如果类没有定义 __str__，print() 会回退使用 __repr__
        """交互环境显示"""
        return f"Vector({self.x}, {self.y})"

    def __add__(self, other):
        # v1 + v2 → Python 解释器翻译为 v1.__add__(v2)
        # self = v1（运算符左侧）, other = v2（运算符右侧）
        # 等价于 Vector.__add__(v1, v2)
        """v1 + v2"""
        return Vector(self.x + other.x, self.y + other.y)

    def __sub__(self, other):
        # v1 - v2 → Python 翻译为 v1.__sub__(v2)
        """v1 - v2"""
        return Vector(self.x - other.x, self.y - other.y)

    def __mul__(self, n):
        # v * n → Python 翻译为 v.__mul__(n)（标量乘法，向量乘以数字）
        # 注意：n * v 不会触发此方法，会触发 n.__mul__(v) 可能返回 TypeError
        # 如果需要支持 n * v，需要额外定义 __rmul__(self, n)
        """v * n"""
        return Vector(self.x * n, self.y * n)

    def __eq__(self, other):
        # v1 == v2 → Python 翻译为 v1.__eq__(v2)
        # 如果没有定义 __eq__，默认使用 object.__eq__，比较 id（内存地址）
        # 两个值相同的向量会因为内存地址不同而被判定为不等
        """v1 == v2"""
        return self.x == other.x and self.y == other.y

    def __abs__(self):
        # abs(v) → Python 翻译为 v.__abs__()
        """abs(v) 模长"""
        return (self.x**2 + self.y**2) ** 0.5

    def __len__(self):
        # len(v) → Python 翻译为 v.__len__()
        # 返回 2 表示这是一个二维向量，每个向量都有两个分量
        """len(v) 维度"""
        return 2

v1 = Vector(3, 4)  # 触发 __init__，v1.x = 3, v1.y = 4
v2 = Vector(1, 2)  # 触发 __init__，v2.x = 1, v2.y = 2

# f-string 隐式调用 str(v1)，str(v1) 触发 Vector.__str__(v1)，返回 "(3, 4)"
print(f"v1 = {v1}")
# v1 + v2 触发 Vector.__add__(v1, v2) 返回 Vector(4, 6)，再触发 __str__ 输出
print(f"v1 + v2 = {v1 + v2}")
# v1 - v2 触发 Vector.__sub__(v1, v2) 返回 Vector(2, 2)
print(f"v1 - v2 = {v1 - v2}")
# v1 * 3 触发 Vector.__mul__(v1, 3) 返回 Vector(9, 12)
print(f"v1 * 3 = {v1 * 3}")
# v1 == v2 触发 Vector.__eq__(v1, v2)，比较 3!=1 和 4!=2 返回 False
print(f"v1 == v2? {v1 == v2}")
# abs(v1) 触发 Vector.__abs__(v1)，计算 √(3²+4²) = 5.0
print(f"|v1| = {abs(v1):.2f}")
# len(v1) 触发 Vector.__len__(v1)，返回 2
print(f"len(v1) = {len(v1)}")

# __repr__ 可以重建对象
# repr(v1) 调用 __repr__ 返回 "Vector(3, 4)"
# eval("Vector(3, 4)") 让 Python 解释器执行这个字符串表达式，创建新 Vector 实例
v3 = eval(repr(v1))
print(f"重建: {v3}")                # print(v3) 触发 __str__ 输出 (3, 4)

## @dataclass — 省掉样板代码

每个普通类都要写 `__init__` 来逐个赋值、写 `__repr__` 来调试、写 `__eq__` 来比较……全是重复劳动。

`@dataclass` 自动帮你生成这些东西，你只需要声明属性：

```python
from dataclasses import dataclass

@dataclass
class Student:
    name: str        # 必填
    age: int         # 必填
    score: float = 0.0  # 有默认值 → 可选
```

自动获得：
- `__init__(self, name, age, score=0.0)`
- `__repr__()` → `Student(name='小明', age=20, score=85.5)`
- `__eq__()` → 逐字段比较

类型注解 `name: str` 不会强制类型检查（Python 是动态类型），它只是给 dataclass 用的元数据。

### @dataclass 到底帮你生成了什么？

`@dataclass` 是一个类装饰器，它在类定义完成后**动态注入**方法。完整的工作流程：

```
1. Python 读到 @dataclass 装饰器
2. @dataclass 扫描类体中的类型注解字段（name: str, age: int, score: float = 0.0）
3. 按字段声明顺序排列，有默认值的放后面
4. 使用 inspect 和 setattr 在运行时动态生成并注入以下方法：
   → __init__()：按字段顺序生成构造函数参数
   → __repr__()：生成如 Student(name='小明', age=20, score=85.5) 的格式
   → __eq__()：逐字段比较（==），所有字段相等才返回 True
   → __hash__()：仅当传入 frozen=True 时才生成
   → __lt__ / __le__ / __gt__ / __ge__：仅当传入 order=True 时才生成
```

你可以用一个简单的实验来验证：

```python
@dataclass
class Point:
    x: int
    y: int

p = Point(1, 2)
print(p)              # Point(x=1, y=2)  → 来自自动生成的 __repr__
print(p == Point(1, 2))  # True → 来自自动生成的 __eq__
```

### @dataclass 的高级参数

```python
@dataclass(frozen=True)   # 不可变：生成 __hash__，阻止字段赋值
class ImmutablePoint:
    x: int
    y: int

p = ImmutablePoint(1, 2)
# p.x = 3  → 抛出 FrozenInstanceError！

@dataclass(order=True)    # 生成比较方法（<, <=, >, >=）
class SortableStudent:
    name: str
    score: float

students = [SortableStudent("小明", 85), SortableStudent("小红", 92)]
students.sort()  # 按 score 排序（字段声明顺序）
```

- **`frozen=True`**：让实例不可变，像 tuple 一样安全。同时自动生成 `__hash__()`，实例可以作为字典键或放入集合
- **`order=True`**：生成全套比较方法，支持排序
- **`slots=True`**（Python 3.10+）：生成 `__slots__` 代替 `__dict__`，节省内存

### __post_init__：在 __init__ 之后执行自定义逻辑

有时候你需要在初始化之后做一些额外处理（比如验证数据、计算派生字段）。`@dataclass` 允许你定义一个 `__post_init__` 方法，它在自动生成的 `__init__` 末尾被调用：

```python
from dataclasses import dataclass
from typing import List

@dataclass
class Student:
    name: str
    scores: List[float]

    def __post_init__(self):
        """在 __init__ 完成后自动调用，用于验证和计算"""
        if any(s < 0 or s > 100 for s in self.scores):
            raise ValueError("成绩必须在 0-100 之间")

@dataclass
class Rectangle:
    width: float
    height: float
    area: float = 0.0  # 默认值会被 __post_init__ 覆盖

    def __post_init__(self):
        self.area = self.width * self.height  # 自动计算面积
```

`__post_init__` 是处理以下场景的利器：
- 字段验证（检查值是否合法）
- 计算派生字段（面积、全名等）
- 转换或标准化输入数据

### 什么时候 @dataclass 不够用？

| 场景 | 替代方案 | 原因 |
|------|---------|------|
| 需要复杂的验证逻辑 | 手写 `__init__` + `__post_init__` | 验证逻辑太多，不适合写在类里 |
| 类有复杂的继承层级 | 手写或用 `attrs` 库 | dataclass 的继承支持有限 |
| 性能极度敏感 | `__slots__` | `@dataclass(slots=True)` 可缓解 |
| 需要 ORM 映射（SQLAlchemy） | ORM 自身声明式语法 | dataclass 与 ORM 集成不完全 |
| django 模型类 | django.db.models.Model | Django 用自己的一套元编程 |
| 字段运行时才确定 | 手写类 | dataclass 需要编译时字段声明 |
| 需要深拷贝、序列化等 | `dataclasses-json`、`marshmallow` | dataclass 本身不包含这些 |
| 实例需要可变但也要可哈希 | 手写 `__hash__` | dataclass 只有在 frozen 模式下才生成 `__hash__` |

### @dataclass vs namedtuple

Python 中还有一个老朋友叫做 `namedtuple`：

```python
from collections import namedtuple
Point = namedtuple('Point', ['x', 'y'])
p = Point(1, 2)
```

| 特性 | @dataclass | namedtuple |
|------|-----------|------------|
| **可变性** | 默认可变（可改字段） | 不可变 |
| **类型注解** | 原生支持 | 需要配合 `typing.NamedTuple` |
| **自定义方法** | 完全支持 | 有限支持 |
| **性能** | 普通（使用 `__dict__`） | 极快（基于 tuple，内存紧凑） |
| **继承** | 做得到但稍复杂 | 支持有限 |
| **默认值** | 灵活 | 语法稍别扭 |
| **适用场景** | 需要行为的"半数据"对象 | 纯数据容器，不可变记录 |

简单规则：**需要逻辑和验证用 `@dataclass`，纯数据传递用 `namedtuple`**。</cell id="d3c521e6">

In [ ]:
# ================================================
# 6. @dataclass — 数据类的优雅写法
# ================================================

from dataclasses import dataclass

# @dataclass 是一个类装饰器，接收类作为参数，返回一个增强后的新类
# 背后原理：@dataclass 在运行时解析类体中的类型注解字段，自动生成以下方法：
#   1. __init__() — 按字段声明顺序生成构造方法参数列表
#      有默认值的字段自动变为可选参数（放在参数列表末尾）
#      对上例生成：__init__(self, name: str, age: int, score: float = 0.0)
#   2. __repr__() — 生成如 Student(name='小明', age=20, score=85.5) 的格式
#   3. __eq__() — 逐字段比较（== 运算符），所有字段值相等才返回 True
#   4. __hash__() — 仅在传入 frozen=True 时生成（使实例可哈希）
#   5. __lt__/__le__/__gt__/__ge__ — 传入 order=True 时生成（排序支持）
# 这些方法由 dataclass 元编程（通过 setattr 注入类体）动态添加
@dataclass
class Student:
    name: str         # 类型注解 str，dataclass 将其识别为一个"字段"（field），成为 __init__ 的必填参数
    age: int          # 同上
    score: float = 0.0  # 有默认值的字段，在 __init__ 中变为可选参数，且必须排在必填参数之后

    # 类型注解 name: str 不会强制类型检查（Python 是动态类型语言）
    # 这些注解只被 dataclass 用作生成方法的元数据，运行时不会校验传入值的类型

    @property
    def grade(self):
        # 计算属性（只读），不占用 dataclass 字段空间，靠实时计算产生
        # @property 定义在 @dataclass 类中完全正常，互不影响
        if self.score >= 90:
            return "A"
        elif self.score >= 80:
            return "B"
        elif self.score >= 60:
            return "C"
        else:
            return "D"

# __init__、__repr__、__eq__ 全部由 @dataclass 自动生成，无需手写
s1 = Student("小明", 20, 85.5)   # 调用自动生成的 __init__，按字段顺序赋值
s2 = Student("小明", 20, 85.5)   # 数据与 s1 完全相同（字段值全部一致）
s3 = Student("小红", 19, 92.0)   # 不同数据

# print(s1) 内部调用 str(s1)，str(s1) 调用自动生成的 __repr__
# 输出：Student(name='小明', age=20, score=85.5)
print(s1)
# s1 == s2 触发自动生成的 __eq__：逐字段比较 name、age、score，全部相等 → True
print(f"s1 == s2: {s1 == s2}")
# s1 == s3：name 不相同（'小明' vs '小红'）→ 返回 False
print(f"s1 == s3: {s1 == s3}")
# s1.grade 触发 @property getter，score=85.5 => 80-90 区间 => B
print(f"{s1.name} 成绩: {s1.grade}")
# s3.grade 触发 @property getter，score=92.0 => >=90 => A
print(f"{s3.name} 成绩: {s3.grade}")

---

## 🎯 本课总结

| 概念 | 核心要点 |
|------|---------|
| **类 vs 对象** | 类是蓝图，对象是实例。`__init__` 构造，`self` 代表"自己" |
| **Python 对象模型** | 一切皆对象，包括类本身。`__dict__` 存属性，`type()` 查类型 |
| **属性查找链** | 实例 `__dict__` → 类 `__dict__` → 父类 `__dict__`，赋值不走链 |
| **类属性 vs 实例属性** | 类属性共享，实例属性独立，实例赋值会遮蔽类属性 |
| **三种方法** | 实例(self)、类方法(cls)、静态方法(无)，背后是描述符协议 |
| **@property** | 描述符协议实现，getter/setter/deleter 控制属性访问，`_` 前缀保护内部 |
| **继承 + MRO** | C3 线性化算法计算 `__mro__`，决定方法查找顺序 |
| **super()** | 按 MRO 找"下一个"类而非"父类"，在多继承下协作 |
| **多态** | 同方法名不同行为 |
| **鸭子类型** | 不查类型，只查方法。Python 用协议（protocol）替代接口（interface） |
| **魔术方法** | Python 操作的底层实现，所有操作符最终调用对应 dunder 方法 |
| **@dataclass** | 自动生成 init/repr/eq，frozen/order/slots 参数控制行为，`__post_init__` 做验证 |

### 关于 Python 类型系统的一句话

Python 的类型系统是**鸭子类型 + 协议**的动态类型系统。它不像 Java 那样在编译时做类型检查，而是在运行时按需查找属性和方法。这带来了极大的灵活性，也意味着**测试覆盖率更重要**——很多在其他语言里编译阶段就能发现的错误，在 Python 中需要靠测试来捕获。

学好这套知识后，你可以：
- **写出可复用的代码**：用继承和 Mixin 复用公共逻辑
- **写出自然的 API**：用魔术方法让自定义类像内置类型一样优雅
- **写出健壮的代码**：用 @property 封装和保护内部状态
- **写出简洁的代码**：用 @dataclass 减少样板代码

---

## 🧪 课后练习：银行账户系统

实现一个完整的银行账户系统，包含：
1. 余额用 `@property` 封装，外部只读
2. 存款、取款带验证（不能负、不能透支）
3. 支持转账到其他账户
4. 记录交易历史
5. 用魔术方法让 print(account) 更友好
6. 用类方法统计总账户数

> **建议**：完成基础版本后，尝试用 `@dataclass` 重构交易记录类（Transaction），看看哪些代码可以简化。然后试着给账户添加利息功能——用继承实现 `SavingsAccount( BankAccount)`，重写取款逻辑，看看 super() 怎么用到你的代码中。</cell id="467e5752">

In [ ]:
# ================================================
# 综合练习：银行账户系统
# ================================================

class BankAccount:
    """银行账户：封装余额、交易、转账"""

    # bank_name 是类属性，存储在 BankAccount.__dict__ 中，所有账户实例共享同一份引用
    bank_name = "Python 银行"
    # _total_accounts 也是类属性，前导下划线表示"仅供内部使用，外部不应直接修改"
    # 每个新账户创建时自增，用于追踪系统中总共创建了多少个账户
    _total_accounts = 0

    def __init__(self, owner, balance=0):
        # self 指向正在被初始化的新 BankAccount 实例
        self.owner = owner          # 实例属性，存储在当前实例的 __dict__ 中，每个账户独有
        self._balance = balance      # 实例属性，前导下划线表示外部应通过 @property 读取，不应直接访问
        self._transactions = []      # 实例属性，每个账户独立的交易记录列表，互不干扰
        BankAccount._total_accounts += 1  # 访问类属性记录总数，每创建一个新实例就自增 1

    @property
    def balance(self):
        """余额（只读）"""
        # @property 装饰器将 balance 方法变成属性读取接口
        # acc.balance 触发此 getter，返回内部的 self._balance
        # 未定义 @balance.setter → 外部只能读取不能赋值，实现真正的"只读"
        return self._balance

    def deposit(self, amount):
        """存款：不能为负"""
        # 实例方法，self 指向调用此方法的那个具体账户对象
        # 例如 a1.deposit(500) 中 self = a1
        if amount <= 0:
            raise ValueError("存款金额必须大于 0")
        self._balance += amount
        # 在实例的 _transactions 列表中追加一条记录
        self._transactions.append(f"存入: +{amount}")
        return self._balance

    def withdraw(self, amount):
        """取款：不能透支"""
        if amount <= 0:
            raise ValueError("取款金额必须大于 0")
        if amount > self._balance:
            raise ValueError("余额不足！")
        self._balance -= amount
        self._transactions.append(f"取出: -{amount}")
        return self._balance

    def transfer(self, amount, target):
        """转账到另一个账户"""
        # self 是转出方，target 是转入方（另一个 BankAccount 实例）
        # 复用已有的 withdraw 和 deposit 方法，贯彻 DRY（Don't Repeat Yourself）
        self.withdraw(amount)   # 从 self 账户扣款（含验证逻辑：余额不足会抛异常）
        target.deposit(amount)  # 向 target 账户存款（含验证逻辑：金额为负会抛异常）
        self._transactions.append(f"转出: {amount} -> {target.owner}")

    def show_history(self):
        """打印交易历史"""
        # self.bank_name 走属性查找：实例 __dict__ 没有 → 类 __dict__ 找到（类属性共享）
        # self.balance 走 @property getter，返回 self._balance
        print(f"\n{'='*35}")
        print(f"  {self.bank_name} — {self.owner}")
        print(f"  余额: {self.balance}")
        print(f"{'='*35}")
        for t in self._transactions:
            print(f"    {t}")
        print(f"{'='*35}")

    @classmethod
    def total_accounts(cls):
        # @classmethod 让方法接收"类"（cls）而非"实例"（self）
        # cls 就是 BankAccount 类对象本身
        # 关键区别：即使被子类调用，cls 也指向子类而非固定写死的 BankAccount
        return cls._total_accounts

    def __str__(self):
        # print(acc) 或 str(acc) 时自动调用
        # 返回用户友好的账户摘要字符串
        return f"[{self.owner}] 余额: {self.balance}"

    def __repr__(self):
        # repr(acc) 或交互式环境显示时自动调用
        # 遵循 Python 惯例：__repr__ 返回的字符串应能通过 eval() 重建对象
        return f"BankAccount('{self.owner}', {self.balance})"


# ========== 运行测试 ==========
print("=== 银行账户演示 ===")

a1 = BankAccount("小明", 1000)  # 触发 __init__，_total_accounts = 1
a2 = BankAccount("小红", 500)   # 触发 __init__，_total_accounts = 2
print(f"总账户: {BankAccount.total_accounts()}")  # 类方法，返回 2

a1.deposit(500)    # self = a1，a1._balance 从 1000 增至 1500
a2.withdraw(200)   # self = a2，a2._balance 从 500 减至 300
a1.transfer(300, a2)  # self = a1, target = a2：a1 扣 300 → 1200，a2 加 300 → 600

a1.show_history()
a2.show_history()

# a1.balance 走 @property getter，返回 self._balance（1200）
print(f"\n小明: {a1.balance}")
# a2.balance 走 @property getter，返回 self._balance（600）
print(f"小红: {a2.balance}")

# print(a1) 触发 __str__，返回 "[小明] 余额: 1200"
print(f"\n账户信息: {a1}")
# repr(a2) 触发 __repr__，返回 "BankAccount('小红', 600)"
print(f"repr: {repr(a2)}")

# 验证
try:
    a1.withdraw(99999)           # 1200 < 99999 → 触发 "余额不足！" 异常
except ValueError as e:
    print(f"\n验证生效: {e}")